# Baby MNIST MEDAL: saved model and encoder attribution

Retrained using the saved winning architecture on the saved 1,149-row training split. Inputs retain their original 0–16 pixel scale. Attribution uses the 288-row validation split, a zero baseline, and 100 integration steps. These are sklearn 8×8 digits. Validation distillation is unavailable because the saved t-SNE teacher contains training rows only. The earlier winning trial metrics do not describe this new fit.


In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import medal
OUT = Path('/Users/clairehe/Documents/GitHub/iicd_MEDAL_feat/outputs/baby_mnist_medal')
cfg = json.loads((OUT / 'config.json').read_text())
model = medal.load_model(OUT / 'baby_mnist_ckpts/final.pt', input_dim=64, hidden_dims=tuple(cfg['hidden_dims']), latent_dim=2, activation=cfg['activation'], use_batchnorm=cfg['use_batchnorm'], dropout_rate=cfg['dropout_rate'])
model.eval()
metrics = json.loads((OUT / 'metrics.json').read_text())
metrics

In [ ]:
data = np.load(OUT / 'encoder_ig.npz')
X_val = data['X']
y_val = data['labels']
attributions = data['attributions']  # (samples, 2 latent coordinates, 64 pixels)
importance = np.abs(attributions).sum(axis=1)
print('Attribution shape:', attributions.shape)
plt.figure(figsize=(15, 4))
plt.imshow(plt.imread(OUT / 'encoder_ig.png'))
plt.axis('off');

In [ ]:
# Inspect a validation example and signed attribution for each coordinate.
i = 0
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(X_val[i].reshape(8,8), cmap='gray')
axes[0].set_title(f'Digit {y_val[i]}')
limit = np.abs(attributions[i]).max()
for j in range(2):
    axes[j+1].imshow(attributions[i,j].reshape(8,8), cmap='coolwarm', vmin=-limit, vmax=limit)
    axes[j+1].set_title(f'IG for latent {j+1}')
for ax in axes: ax.axis('off')
plt.show()

In [ ]:
# Adapted from benchmarking/feature_attribution.ipynb: L2 across latent coordinates.
l2_importance = np.linalg.norm(attributions, axis=1)
fig, axes = plt.subplots(2, 5, figsize=(10, 4), layout='constrained')
for digit, ax in enumerate(axes.flat):
    values = l2_importance[y_val == digit].mean(axis=0)
    values = values / max(values.sum(), np.finfo(float).eps)
    ax.imshow(values.reshape(8, 8), cmap='turbo', vmin=0)
    ax.set_title(str(digit))
    ax.axis('off')
fig.suptitle('Encoder IG: mean L2 magnitude, normalized within each digit')
plt.show()
